# AI Video Highlight Generator — Google Colab

**Before running:** `Runtime → Change runtime type → T4 GPU`

Run each cell top to bottom. First-time setup takes ~5 min (model downloads).

## Cell 1 — Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU : {gpu}')
    print(f'VRAM: {vram:.1f} GB')
else:
    raise SystemExit('No GPU found — go to Runtime → Change runtime type → T4 GPU')

## Cell 2 — Install system dependencies (ffmpeg + Ollama)

In [ ]:
!apt-get install -qq ffmpeg
!curl -fsSL https://ollama.com/install.sh | sh
print('ffmpeg + Ollama installed')

## Cell 3 — Start Ollama server

In [ ]:
import subprocess, time, requests

subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for i in range(30):
    try:
        requests.get('http://localhost:11434', timeout=2)
        print(f'Ollama ready ({i+1}s)')
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Ollama failed to start')

## Cell 4 — Pull AI models

| Model | Size | Role |
|---|---|---|
| `llama3.2` | ~2 GB | Explainer + Highlighter |
| `llava` | ~4.7 GB | Visual frame description |


In [ ]:
import subprocess

for model in ['llama3.2', 'llava']:
    print(f'Pulling {model} ...')
    r = subprocess.run(['ollama', 'pull', model], capture_output=True, text=True)
    if r.returncode == 0:
        print(f'  {model} ready')
    else:
        print(f'  ERROR: {r.stderr[-300:]}')

## Cell 5 — Install Python packages

In [ ]:
!pip install -q openai-whisper requests numpy
print('Done')

## Cell 6 — Clone repo from GitHub

In [ ]:
import os

REPO_URL = 'https://github.com/rishiboppana/video_editing_agent.git'
LOCAL_PATH = '/content/video_editing_agent'

if os.path.exists(LOCAL_PATH):
    # Pull latest changes if already cloned
    !git -C {LOCAL_PATH} pull
else:
    !git clone {REPO_URL} {LOCAL_PATH}

os.chdir(LOCAL_PATH)
os.makedirs('videos', exist_ok=True)
os.makedirs('output', exist_ok=True)
print(f'Ready: {os.getcwd()}')

## Cell 7 — Upload your video

In [ ]:
from google.colab import files
import os

print('Select your video file...')
uploaded = files.upload()

for name, data in uploaded.items():
    dest = f'videos/{name}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'Saved → {dest}')

VIDEO_FILE = f"videos/{list(uploaded.keys())[0]}"
print(f'\nVIDEO_FILE = "{VIDEO_FILE}"')

## Cell 8 — Run the pipeline

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

DURATION = 30  # highlight reel length in seconds

!python3 main.py "{VIDEO_FILE}" --duration {DURATION}

## Cell 9 — Download the highlight video

In [ ]:
import glob, os
from google.colab import files

outputs = sorted(glob.glob('output/*.mp4'), key=os.path.getmtime)
if not outputs:
    print('No output found — check logs in Cell 8')
else:
    print(f'Downloading: {outputs[-1]}')
    files.download(outputs[-1])